# Exploratory Data Analysis (EDA)

Inspect the shape of our dataset

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

train_df = pd.read_csv('data/UNSW_NB15_training-set.csv')
test_df = pd.read_csv('data/UNSW_NB15_testing-set.csv')

train_df.drop('id', axis=1, inplace=True, errors='ignore')
test_df.drop('id', axis=1, inplace=True, errors='ignore')

print(f"Training set shape: {train_df.shape}")
print(f"Testing set shape: {test_df.shape}")

print("\n--- Training Set Info ---")
train_df.info()

print("\n--- Attack Category Distribution ---")
print(train_df['attack_cat'].value_counts())

print("\n--- Binary Target Distribution ---")
print(train_df['label'].value_counts())

Training set shape: (175341, 44)
Testing set shape: (82332, 44)

--- Training Set Info ---
<class 'pandas.DataFrame'>
RangeIndex: 175341 entries, 0 to 175340
Data columns (total 44 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   dur                175341 non-null  float64
 1   proto              175341 non-null  str    
 2   service            175341 non-null  str    
 3   state              175341 non-null  str    
 4   spkts              175341 non-null  int64  
 5   dpkts              175341 non-null  int64  
 6   sbytes             175341 non-null  int64  
 7   dbytes             175341 non-null  int64  
 8   rate               175341 non-null  float64
 9   sttl               175341 non-null  int64  
 10  dttl               175341 non-null  int64  
 11  sload              175341 non-null  float64
 12  dload              175341 non-null  float64
 13  sloss              175341 non-null  int64  
 14  dloss              1

Based on the type of data, we'll use sample weight bias to address the class imbalance

In [2]:
import numpy as np
from sklearn.utils.class_weight import compute_sample_weight

# The UNSW dataset often has trailing spaces in the attack_cat column
train_df['attack_cat'] = train_df['attack_cat'].str.strip()
test_df['attack_cat'] = test_df['attack_cat'].str.strip()

# Compute sample weights for the multi-class target to penalize majority class bias
sample_weights = compute_sample_weight(
    class_weight='balanced',
    y=train_df['attack_cat']
)

print(f"Sample weights shape: {sample_weights.shape}")
print(f"Unique weight values: {np.unique(sample_weights)}")

Sample weights shape: (175341,)
Unique weight values: [  0.31310893   0.4383525    0.5250831    0.96425979   1.42972114
   1.67134687   8.76705     10.04243986  15.47581642 134.87769231]


## Categorical Encoding

one hot encoding, with all unique protocols grouped together

In [3]:
cat_cols = ['proto', 'service', 'state']

# Keep the top 10 protocols, label the rest as 'other' to prevent dimensionality explosion
top_protos = train_df['proto'].value_counts().nlargest(10).index
train_df['proto'] = train_df['proto'].where(train_df['proto'].isin(top_protos), 'other')
test_df['proto'] = test_df['proto'].where(test_df['proto'].isin(top_protos), 'other')

# Apply one-hot encoding
train_encoded = pd.get_dummies(train_df, columns=cat_cols)
test_encoded = pd.get_dummies(test_df, columns=cat_cols)

# Align columns to ensure the test set matches the training set features perfectly
train_encoded, test_encoded = train_encoded.align(test_encoded, join='inner', axis=1)

print(f"Encoded training set shape: {train_encoded.shape}")
print(f"Encoded testing set shape: {test_encoded.shape}")

Encoded training set shape: (175341, 70)
Encoded testing set shape: (82332, 70)


## Normalization
Probably optional for XGBoost, but might be needed if we decide to try KNN or other models

In [4]:
from sklearn.preprocessing import StandardScaler

# Separate targets from features
targets = ['label', 'attack_cat']

X_train = train_encoded.drop(columns=targets)
y_train_multi = train_encoded['attack_cat']
y_train_binary = train_encoded['label']

X_test = test_encoded.drop(columns=targets)
y_test_multi = test_encoded['attack_cat']
y_test_binary = test_encoded['label']

# Identify continuous columns (excluding the one-hot encoded boolean/uint8 columns)
continuous_cols = [col for col in X_train.columns if not any(col.startswith(prefix) for prefix in ['proto_', 'service_', 'state_'])]

# Initialize the scaler
scaler = StandardScaler()

# Fit ONLY on the training data to prevent data leakage, then transform both
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

print("Scaling complete.")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

Scaling complete.
X_train shape: (175341, 68)
X_test shape: (82332, 68)


# XGBoost Training

In [5]:
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from sklearn.metrics import classification_report

# XGBoost strictly requires numerical labels starting from 0
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train_multi)
y_test_enc = le.transform(y_test_multi)

# Initialize the classifier
xgb_model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=len(le.classes_),
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1 
)

print("Training XGBoost model... this may take a minute.")

# Fit the model using the sample weights to handle the class imbalance
xgb_model.fit(
    X_train, 
    y_train_enc, 
    sample_weight=sample_weights
)

# Generate predictions
y_pred_enc = xgb_model.predict(X_test)

# Convert integer predictions back to original string labels for readable metrics
y_pred = le.inverse_transform(y_pred_enc)

print("\n--- Classification Report ---")
print(classification_report(y_test_multi, y_pred, digits=4))

Training XGBoost model... this may take a minute.

--- Classification Report ---
                precision    recall  f1-score   support

      Analysis     0.0382    0.1699    0.0624       677
      Backdoor     0.0594    0.6690    0.1091       583
           DoS     0.4397    0.2230    0.2960      4089
      Exploits     0.8066    0.6020    0.6895     11132
       Fuzzers     0.2519    0.6617    0.3649      6062
       Generic     0.9982    0.9703    0.9841     18871
        Normal     0.9866    0.6118    0.7552     37000
Reconnaissance     0.8641    0.8475    0.8557      3496
     Shellcode     0.2185    0.9709    0.3567       378
         Worms     0.6607    0.8409    0.7400        44

      accuracy                         0.6856     82332
     macro avg     0.5324    0.6567    0.5214     82332
  weighted avg     0.8604    0.6856    0.7394     82332



Results aren't great, implementing feature selection and changing the way the weights are calculated

In [6]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import classification_report

# 1. Feature Importance Pruning (Relaxed)
importances = xgb_model.feature_importances_
feature_names = X_train.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Keep top 50 features instead of 20 to retain subtle network signatures
top_features = feature_importance_df.head(50)['Feature'].tolist()
X_train_pruned = X_train[top_features]
X_test_pruned = X_test[top_features]

print(f"Reduced features from {X_train.shape[1]} to {X_train_pruned.shape[1]}")

# 2. Custom Class Weighting (Clipping)
class_counts = pd.Series(y_train_enc).value_counts().sort_index()
total_samples = len(y_train_enc)
num_classes = len(class_counts)

raw_weights = total_samples / (num_classes * class_counts)

# Clip the weights to prevent extreme penalties without flattening them entirely
clipped_weights = np.clip(raw_weights, a_min=1.0, a_max=15.0)

sample_weights_custom = np.array([clipped_weights[label] for label in y_train_enc])

# 3. Regularization and Retraining (Relaxed)
xgb_model_tuned = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=len(le.classes_),
    eval_metric='mlogloss',
    max_depth=10,          
    learning_rate=0.1,     
    gamma=0.1,             
    reg_lambda=2.0,        
    subsample=0.8,        
    colsample_bytree=0.8, 
    random_state=42,
    n_jobs=-1
)

print("Training relaxed XGBoost model...")

xgb_model_tuned.fit(
    X_train_pruned, 
    y_train_enc, 
    sample_weight=sample_weights_custom
)

y_pred_enc_tuned = xgb_model_tuned.predict(X_test_pruned)
y_pred_tuned = le.inverse_transform(y_pred_enc_tuned)

print("\n--- Relaxed Classification Report ---")
print(classification_report(y_test_multi, y_pred_tuned, digits=4))

Reduced features from 68 to 50
Training relaxed XGBoost model...

--- Relaxed Classification Report ---
                precision    recall  f1-score   support

      Analysis     0.0445    0.2230    0.0743       677
      Backdoor     0.0513    0.5197    0.0934       583
           DoS     0.5243    0.1372    0.2175      4089
      Exploits     0.7473    0.6665    0.7046     11132
       Fuzzers     0.3039    0.5371    0.3882      6062
       Generic     0.9981    0.9723    0.9850     18871
        Normal     0.9627    0.7275    0.8287     37000
Reconnaissance     0.9022    0.8289    0.8640      3496
     Shellcode     0.2087    0.9471    0.3421       378
         Worms     0.5965    0.7727    0.6733        44

      accuracy                         0.7317     82332
     macro avg     0.5340    0.6332    0.5171     82332
  weighted avg     0.8512    0.7317    0.7727     82332



# Isolation Forest

In [7]:
from sklearn.ensemble import IsolationForest
import numpy as np
from sklearn.metrics import classification_report

iso_forest = IsolationForest(
    n_estimators=100,
    contamination='auto',
    random_state=42,
    n_jobs=-1
)

print("Training Isolation Forest...")
iso_forest.fit(X_train_pruned)

y_pred_iso_raw = iso_forest.predict(X_test_pruned)

# -1 represents outliers, 1 represents inliers.
# We map -1 to 1 (Attack) and 1 to 0 (Normal) to match y_test_binary.
y_pred_iso = np.where(y_pred_iso_raw == -1, 1, 0)

print("\n--- Isolation Forest Classification Report (Binary) ---")
print(classification_report(y_test_binary, y_pred_iso, digits=4))

Training Isolation Forest...

--- Isolation Forest Classification Report (Binary) ---
              precision    recall  f1-score   support

           0     0.4499    0.9689    0.6144     37000
           1     0.5647    0.0329    0.0622     45332

    accuracy                         0.4536     82332
   macro avg     0.5073    0.5009    0.3383     82332
weighted avg     0.5131    0.4536    0.3104     82332



# Random Forest and MLP

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

# 1. Random Forest Implementation
print("Training Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced_subsample',
    max_depth=15, 
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_pruned, y_train_enc)

y_pred_enc_rf = rf_model.predict(X_test_pruned)
y_pred_rf = le.inverse_transform(y_pred_enc_rf)

print("\n--- Random Forest Classification Report ---")
print(classification_report(y_test_multi, y_pred_rf, digits=4))


# 2. Multi-Layer Perceptron (MLP) Implementation
print("\nTraining Multi-Layer Perceptron (MLP)...")
mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    alpha=0.001, 
    batch_size=256,
    max_iter=200,
    early_stopping=True,
    random_state=42
)

# MLP in sklearn does not accept sample_weight in fit()
mlp_model.fit(X_train_pruned, y_train_enc)

y_pred_enc_mlp = mlp_model.predict(X_test_pruned)
y_pred_mlp = le.inverse_transform(y_pred_enc_mlp)

print("\n--- MLP Classification Report ---")
print(classification_report(y_test_multi, y_pred_mlp, digits=4))

Training Random Forest...

--- Random Forest Classification Report ---
                precision    recall  f1-score   support

      Analysis     0.0092    0.0355    0.0146       677
      Backdoor     0.0397    0.3774    0.0719       583
           DoS     0.3373    0.2294    0.2731      4089
      Exploits     0.7997    0.6148    0.6952     11132
       Fuzzers     0.2441    0.6897    0.3606      6062
       Generic     0.9998    0.9636    0.9814     18871
        Normal     0.9931    0.5850    0.7363     37000
Reconnaissance     0.8516    0.8473    0.8494      3496
     Shellcode     0.1653    0.9550    0.2818       378
         Worms     0.5238    0.7500    0.6168        44

      accuracy                         0.6728     82332
     macro avg     0.4964    0.6048    0.4881     82332
  weighted avg     0.8559    0.6728    0.7282     82332


Training Multi-Layer Perceptron (MLP)...

--- MLP Classification Report ---
                precision    recall  f1-score   support

      An

# Resplit Data

Random Forest, MLP, and XGBoost got similar results, which would indicate a problem with the dataset. 

In [9]:
from sklearn.model_selection import train_test_split

# 1. Merge the raw datasets
combined_df = pd.concat([train_df, test_df], ignore_index=True)

# 2. Separate features and targets before splitting
X_combined = combined_df.drop(columns=['label', 'attack_cat'])
y_combined_multi = combined_df['attack_cat']
y_combined_binary = combined_df['label']

# 3. Perform a stratified split (80% train, 20% test)
X_train_raw, X_test_raw, y_train_multi, y_test_multi = train_test_split(
    X_combined, y_combined_multi, 
    test_size=0.20, 
    stratify=y_combined_multi, 
    random_state=42
)

y_train_binary = combined_df.loc[y_train_multi.index, 'label']
y_test_binary = combined_df.loc[y_test_multi.index, 'label']

print(f"New Stratified Training Set Shape: {X_train_raw.shape}")
print(f"New Stratified Testing Set Shape: {X_test_raw.shape}")

# 4. Re-apply the Encoding Pipeline
cat_cols = ['proto', 'service', 'state']
X_train_enc = pd.get_dummies(X_train_raw, columns=cat_cols)
X_test_enc = pd.get_dummies(X_test_raw, columns=cat_cols)
X_train_enc, X_test_enc = X_train_enc.align(X_test_enc, join='inner', axis=1)

# 5. Re-apply the Scaling Pipeline
continuous_cols = [col for col in X_train_enc.columns if not any(col.startswith(prefix) for prefix in ['proto_', 'service_', 'state_'])]
scaler = StandardScaler()
X_train_enc[continuous_cols] = scaler.fit_transform(X_train_enc[continuous_cols])
X_test_enc[continuous_cols] = scaler.transform(X_test_enc[continuous_cols])

# Overwrite X_train and X_test so your previous model cells work without modification
X_train = X_train_enc
X_test = X_test_enc

print("Data successfully merged, stratified, encoded, and scaled.")

New Stratified Training Set Shape: (206138, 42)
New Stratified Testing Set Shape: (51535, 42)
Data successfully merged, stratified, encoded, and scaled.


In [10]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

# 1. Recalculate Label Encodings for the new stratified split
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train_multi)
y_test_enc = le.transform(y_test_multi)

# 2. Train a fast baseline model to extract correct feature importances
print("Training baseline XGBoost to extract feature importances...")
xgb_base = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=len(le.classes_),
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)
xgb_base.fit(X_train, y_train_enc)

# 3. Feature Importance Pruning (Relaxed)
importances = xgb_base.feature_importances_
feature_names = X_train.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

top_features = feature_importance_df.head(50)['Feature'].tolist()
X_train_pruned = X_train[top_features]
X_test_pruned = X_test[top_features]

print(f"Reduced features from {X_train.shape[1]} to {X_train_pruned.shape[1]}")

# 4. Custom Class Weighting (Clipping) recalculated for new row count
class_counts = pd.Series(y_train_enc).value_counts().sort_index()
total_samples = len(y_train_enc)
num_classes = len(class_counts)

raw_weights = total_samples / (num_classes * class_counts)
clipped_weights = np.clip(raw_weights, a_min=1.0, a_max=15.0)

sample_weights_custom = np.array([clipped_weights[label] for label in y_train_enc])

# 5. Regularization and Retraining
xgb_model_tuned = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=len(le.classes_),
    eval_metric='mlogloss',
    max_depth=10,          
    learning_rate=0.1,     
    gamma=0.1,             
    reg_lambda=2.0,        
    subsample=0.8,        
    colsample_bytree=0.8, 
    random_state=42,
    n_jobs=-1
)

print("Training relaxed XGBoost model on stratified data...")

xgb_model_tuned.fit(
    X_train_pruned, 
    y_train_enc, 
    sample_weight=sample_weights_custom
)

y_pred_enc_tuned = xgb_model_tuned.predict(X_test_pruned)
y_pred_tuned = le.inverse_transform(y_pred_enc_tuned)

print("\n--- Relaxed Classification Report (Stratified) ---")
print(classification_report(y_test_multi, y_pred_tuned, digits=4))

Training baseline XGBoost to extract feature importances...
Reduced features from 70 to 50
Training relaxed XGBoost model on stratified data...

--- Relaxed Classification Report (Stratified) ---
                precision    recall  f1-score   support

      Analysis     0.0717    0.1869    0.1036       535
      Backdoor     0.0444    0.2146    0.0735       466
           DoS     0.3697    0.4998    0.4250      3271
      Exploits     0.8533    0.5591    0.6756      8905
       Fuzzers     0.7431    0.5768    0.6495      4849
       Generic     0.9964    0.9826    0.9894     11774
        Normal     0.9177    0.9375    0.9275     18600
Reconnaissance     0.8747    0.8031    0.8373      2798
     Shellcode     0.4377    0.9073    0.5905       302
         Worms     0.4655    0.7714    0.5806        35

      accuracy                         0.7988     51535
     macro avg     0.5774    0.6439    0.5853     51535
  weighted avg     0.8512    0.7988    0.8167     51535



In [11]:
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report
import numpy as np

print("=========================================")
print("1. Training Isolation Forest (Binary)")
print("=========================================")

# Isolation Forest is unsupervised, but we train it on the pruned features for consistency
iso_forest = IsolationForest(
    n_estimators=100,
    contamination='auto',
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(X_train_pruned)
y_pred_iso_raw = iso_forest.predict(X_test_pruned)

# Map -1 (outlier) to 1 (Attack) and 1 (inlier) to 0 (Normal)
y_pred_iso = np.where(y_pred_iso_raw == -1, 1, 0)

print("\n--- Isolation Forest Classification Report ---")
print(classification_report(y_test_binary, y_pred_iso, digits=4))


print("\n=========================================")
print("2. Training Random Forest (Multiclass)")
print("=========================================")

rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced_subsample',
    max_depth=15, 
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_pruned, y_train_enc)

y_pred_enc_rf = rf_model.predict(X_test_pruned)
y_pred_rf = le.inverse_transform(y_pred_enc_rf)

print("\n--- Random Forest Classification Report ---")
print(classification_report(y_test_multi, y_pred_rf, digits=4))


print("\n=========================================")
print("3. Training Multi-Layer Perceptron (Multiclass)")
print("=========================================")

mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    alpha=0.001, 
    batch_size=256,
    max_iter=200,
    early_stopping=True,
    random_state=42
)

mlp_model.fit(X_train_pruned, y_train_enc)

y_pred_enc_mlp = mlp_model.predict(X_test_pruned)
y_pred_mlp = le.inverse_transform(y_pred_enc_mlp)

print("\n--- MLP Classification Report ---")
print(classification_report(y_test_multi, y_pred_mlp, digits=4))


print("\n=========================================")
print("4. XGBoost Binary Roll-up (For Direct Comparison)")
print("=========================================")

# Assuming xgb_model_tuned is still in memory from the previous cell execution
normal_enc = le.transform(['Normal'])[0]

# 0 if Normal, 1 if any Attack
y_pred_xgb_binary = np.where(y_pred_enc_tuned == normal_enc, 0, 1)

print("\n--- XGBoost Classification Report (Binary Roll-up) ---")
print(classification_report(y_test_binary, y_pred_xgb_binary, digits=4))

1. Training Isolation Forest (Binary)

--- Isolation Forest Classification Report ---
              precision    recall  f1-score   support

           0     0.3573    0.9671    0.5218     18600
           1     0.4879    0.0177    0.0342     32935

    accuracy                         0.3604     51535
   macro avg     0.4226    0.4924    0.2780     51535
weighted avg     0.4408    0.3604    0.2102     51535


2. Training Random Forest (Multiclass)

--- Random Forest Classification Report ---
                precision    recall  f1-score   support

      Analysis     0.0869    0.2355    0.1270       535
      Backdoor     0.0543    0.2039    0.0857       466
           DoS     0.3600    0.6243    0.4566      3271
      Exploits     0.8615    0.5282    0.6549      8905
       Fuzzers     0.4826    0.8385    0.6126      4849
       Generic     0.9997    0.9747    0.9870     11774
        Normal     0.9952    0.7304    0.8425     18600
Reconnaissance     0.8367    0.8256    0.8311      27